# 04 - MONAI Introduction

**Goal:** Learn MONAI - the framework your production code uses.

---

## Before Running This Notebook

MONAI needs to be installed. Update `requirements.txt`:

```
# Uncomment this line:
monai==1.4.0
```

Then rebuild Docker:
```bash
docker compose down
docker compose up --build
```

## What is MONAI?

**Medical Open Network for AI** - PyTorch library for medical imaging.

Built by NVIDIA + academic partners. Provides:
- Medical imaging transforms (3D, DICOM handling)
- Pre-built networks (UNet, SwinUNETR, etc.)
- Loss functions (Dice, focal, etc.)
- Sliding window inference
- Medical image I/O

Your production code uses MONAI extensively.

In [ ]:
# Check if MONAI is installed
try:
    import monai
    print(f"MONAI version: {monai.__version__}")
    monai.config.print_config()
except ImportError:
    print("MONAI not installed!")
    print("Uncomment 'monai==1.4.0' in requirements.txt and rebuild Docker.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# MONAI imports
from monai.transforms import (
    Compose, LoadImage, EnsureChannelFirst, ScaleIntensity,
    RandFlip, RandRotate, RandZoom, ToTensor
)
from monai.networks.nets import UNet, SwinUNETR
from monai.losses import DiceLoss, DiceCELoss
from monai.inferers import sliding_window_inference
from monai.data import ArrayDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## MONAI Transforms

MONAI transforms handle medical imaging specifics:
- 3D volumes
- Consistent handling of image + label
- Medical-specific augmentations

In [ ]:
# Create synthetic 3D data for demo
def create_synthetic_volume(size=64):
    """Create a synthetic 3D volume with a sphere."""
    volume = np.random.normal(0.2, 0.05, (size, size, size)).astype(np.float32)
    mask = np.zeros((size, size, size), dtype=np.float32)
    
    # Add a sphere
    center = size // 2
    radius = size // 4
    z, y, x = np.ogrid[:size, :size, :size]
    sphere = np.sqrt((x - center)**2 + (y - center)**2 + (z - center)**2) <= radius
    
    volume[sphere] = 0.8
    mask[sphere] = 1.0
    
    return volume, mask

volume, mask = create_synthetic_volume()
print(f"Volume shape: {volume.shape}")
print(f"Mask shape: {mask.shape}")

In [ ]:
# Visualize a slice
slice_idx = 32

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(volume[slice_idx], cmap='gray')
axes[0].set_title(f'Volume (slice {slice_idx})')
axes[0].axis('off')

axes[1].imshow(mask[slice_idx], cmap='Blues')
axes[1].set_title('Mask')
axes[1].axis('off')

axes[2].imshow(volume[slice_idx], cmap='gray')
axes[2].imshow(mask[slice_idx], cmap='Reds', alpha=0.5)
axes[2].set_title('Overlay')
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# MONAI transforms pipeline
from monai.transforms import (
    EnsureChannelFirst, ScaleIntensity, RandFlip, RandRotate90, ToTensor
)

# Training transforms (with augmentation)
train_transforms = Compose([
    EnsureChannelFirst(),        # Add channel dimension: (64,64,64) -> (1,64,64,64)
    ScaleIntensity(),            # Normalize to [0,1]
    RandFlip(prob=0.5, spatial_axis=0),   # Random flip
    RandRotate90(prob=0.5, spatial_axes=(1, 2)),  # Random 90° rotation
    ToTensor()
])

# Apply transforms
volume_tensor = train_transforms(volume)
print(f"After transforms: {volume_tensor.shape}")

## MONAI Networks

Pre-built architectures ready to use:

In [ ]:
# MONAI UNet (2D)
unet_2d = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=3,
    channels=(32, 64, 128, 256),
    strides=(2, 2, 2),
)

# Test
x = torch.randn(1, 1, 128, 128)
print(f"MONAI 2D UNet")
print(f"  Input: {x.shape}")
print(f"  Output: {unet_2d(x).shape}")
print(f"  Parameters: {sum(p.numel() for p in unet_2d.parameters()):,}")

In [ ]:
# MONAI UNet (3D) - like your production code
unet_3d = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=4,  # 4 classes
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
)

# Test with volume like your production input
x = torch.randn(1, 1, 64, 64, 64)
print(f"MONAI 3D UNet")
print(f"  Input: {x.shape}")
print(f"  Output: {unet_3d(x).shape}")
print(f"  Parameters: {sum(p.numel() for p in unet_3d.parameters()):,}")

In [ ]:
# SwinUNETR - what your production code uses!
# (This matches your config)

swin_unetr = SwinUNETR(
    img_size=(96, 96, 96),
    in_channels=1,
    out_channels=4,
    feature_size=24,  # Same as your production config
    use_checkpoint=False,
)

# Test
x = torch.randn(1, 1, 96, 96, 96)
print(f"SwinUNETR (your production model)")
print(f"  Input: {x.shape}")
print(f"  Output: {swin_unetr(x).shape}")
print(f"  Parameters: {sum(p.numel() for p in swin_unetr.parameters()):,}")

## MONAI Loss Functions

In [ ]:
# Dice Loss - handles class imbalance well
dice_loss = DiceLoss(to_onehot_y=True, softmax=True)

# DiceCE Loss - Dice + CrossEntropy combined
dice_ce_loss = DiceCELoss(to_onehot_y=True, softmax=True)

# Example usage
pred = torch.randn(2, 4, 64, 64, 64)  # batch=2, 4 classes, 64^3
target = torch.randint(0, 4, (2, 1, 64, 64, 64))  # Integer labels

loss = dice_ce_loss(pred, target)
print(f"DiceCE Loss: {loss.item():.4f}")

## Sliding Window Inference

For large volumes that don't fit in GPU memory, process in patches:

In [ ]:
# This is exactly what your production code does!

# Large input that might not fit in memory
large_input = torch.randn(1, 1, 192, 192, 192)

# Small model for demo
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=4,
    channels=(8, 16, 32),
    strides=(2, 2),
)

model.eval()
with torch.no_grad():
    # Process in 64x64x64 patches with 50% overlap
    output = sliding_window_inference(
        inputs=large_input,
        roi_size=(64, 64, 64),  # Patch size
        sw_batch_size=2,        # Patches per batch
        predictor=model,
        overlap=0.5,            # 50% overlap between patches
    )

print(f"Input: {large_input.shape}")
print(f"Output: {output.shape}")
print("\nProcessed large volume in patches - same as your production code!")

## Your Production Code Pattern

From `segmentation/src/inference_engine_v016.py`:

```python
from monai.networks.nets import SwinUNETR
from monai.inferers import sliding_window_inference

# Create model
model = SwinUNETR(
    img_size=config["model"]["input_shape"],
    in_channels=1,
    out_channels=nb_classes,
    feature_size=24,
)

# Load weights
model.load_state_dict(torch.load(weights_path))
model = model.to(device)
model.eval()

# Inference
with torch.no_grad():
    with torch.amp.autocast('cuda'):  # Mixed precision
        pred = sliding_window_inference(
            data["image"].unsqueeze(0),
            roi_size=config["model"]["input_shape"],
            sw_batch_size=4,
            predictor=model,
            overlap=0.5,
        )
```

Now you understand every line!

## Summary

| MONAI Component | What it does | Your production usage |
|-----------------|-------------|----------------------|
| **Transforms** | Data preprocessing/augmentation | Normalize, orient, resample |
| **SwinUNETR** | Transformer + U-Net architecture | Main model |
| **DiceLoss** | Overlap-based loss | Training |
| **sliding_window_inference** | Process large volumes in patches | Inference |

**Key insight:** MONAI handles the medical imaging specifics so you focus on the problem, not the plumbing.

---

## Phase 2 Complete!

You now understand:
- What segmentation is (per-pixel classification)
- U-Net architecture (encoder-decoder + skip connections)
- How to train a segmentation model
- MONAI framework (what your production uses)

**Next: Phase 3** - Transformers and SwinUNETR (understanding the "Swin" part).